In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Configuration 
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# SOM Hyperparameters
SOM_WIDTH = 10
SOM_HEIGHT = 10
LATENT_DIM = 64      # Compressed dimension (Autoencoder output)
AE_EPOCHS = 150       # Autoencoder pre-training epochs
SOM_EPOCHS = 100     # SOM training epochs
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

# ==========================================
# 2. Data  
# ==========================================
data_df = pd.read_csv('GSE33000_Top10000_Var.csv', index_col=0, engine='python')

# Separate features and labels
X_raw = data_df.iloc[:, :-1].values
y_raw = data_df.iloc[:, -1].values

y_raw = np.where((y_raw == 'AD') | (y_raw == 'HD'), 'A', y_raw)

# 1. Label Encoding 
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

# 2. Feature Scaling 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

dataset = TensorDataset(
    torch.FloatTensor(X_scaled).to(device), 
    torch.LongTensor(y_encoded).to(device)
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ==========================================
# 3. Deep Learning 
# ==========================================

class Autoencoder(nn.Module):
    """
    Compresses 10,000 features down to LATENT_DIM (e.g., 64).
    This extracts non-linear gene patterns before the SOM.
    """
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim) # Latent representation
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

class SOM(nn.Module):
    """
    PyTorch implementation of a Self-Organizing Map.
    Optimized for GPU batch processing.
    """
    def __init__(self, m, n, dim, n_iter, alpha=None, sigma=None):
        super(SOM, self).__init__()
        self.m = m
        self.n = n
        self.dim = dim
        self.n_iter = n_iter
        self.alpha = alpha if alpha else 0.3
        self.sigma = sigma if sigma else max(m, n) / 2.0
        
        # Initialize weights randomly
        self.weights = nn.Parameter(torch.randn(m * n, dim), requires_grad=False)
        
        # Pre-compute grid locations for neighborhood calculation
        self.locations = torch.LongTensor(np.array(list(self.neuron_locations()))).to(device)
        self.pdist = nn.PairwiseDistance(p=2)

    def neuron_locations(self):
        for i in range(self.m):
            for j in range(self.n):
                yield np.array([i, j])

    def forward(self, x, it):
        # x shape: (batch_size, dim)
        # weights shape: (num_neurons, dim)
        
        # 1. Calculate distances between inputs and weights
        dists = torch.cdist(x, self.weights) # Shape: (batch, num_neurons)
        
        # 2. Find Best Matching Units (BMU)
        _, bmu_indices = torch.min(dists, dim=1)
        
        # 3. Update Weights (if in training mode)
        if self.training:
            bmu_locs = self.locations[bmu_indices] # (batch, 2)
            
            # Decay parameters
            alpha_t = self.alpha * (1 - it / self.n_iter)
            sigma_t = self.sigma * (1 - it / self.n_iter)
            
            # Calculate neighborhood function for all neurons against all batch BMUs
            
            for i in range(x.size(0)): # Iterate batch
                bmu_loc = bmu_locs[i]
                input_vec = x[i]
                
                # Grid distances from BMU
                grid_dists = torch.sum((self.locations - bmu_loc).pow(2), dim=1).float()
                
                # Neighborhood function (Gaussian)
                neighborhood = torch.exp(-grid_dists / (2 * (sigma_t ** 2)))
                
                # Update rule: W = W + alpha * neighborhood * (X - W)
                # Influence shape: (num_neurons, 1)
                influence = (alpha_t * neighborhood).unsqueeze(1)
                self.weights.data += influence * (input_vec - self.weights.data)
                
        return bmu_indices

# ==========================================
# 4. Training 
# ==========================================

#  Phase 1: Train Autoencoder 
print(f"\nPhase 1: Training Autoencoder ({AE_EPOCHS} epochs)")
input_dim = X_scaled.shape[1]
ae_model = Autoencoder(input_dim, LATENT_DIM).to(device)
ae_optimizer = optim.Adam(ae_model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

for epoch in range(AE_EPOCHS):
    total_loss = 0
    for batch_x, _ in dataloader:
        ae_optimizer.zero_grad()
        _, decoded = ae_model(batch_x)
        loss = criterion(decoded, batch_x)
        loss.backward()
        ae_optimizer.step()
        total_loss += loss.item()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{AE_EPOCHS}, Loss: {total_loss / len(dataloader):.4f}")

#  Phase 2: Extract Latent Features 
ae_model.eval()
with torch.no_grad():
    X_tensor = torch.FloatTensor(X_scaled).to(device)
    X_latent, _ = ae_model(X_tensor)
    
#  Phase 3: Train SOM on Latent Features 
print(f"\nPhase 2: Training SOM ({SOM_EPOCHS} epochs)")
som = SOM(m=SOM_WIDTH, n=SOM_HEIGHT, dim=LATENT_DIM, n_iter=SOM_EPOCHS*len(dataloader)).to(device)
som.train()

iter_count = 0
for epoch in range(SOM_EPOCHS):
    # Shuffle latent data for SOM training
    indices = torch.randperm(X_latent.size(0))
    X_latent_shuffled = X_latent[indices]
    
    # Process in batches
    for i in range(0, X_latent.size(0), BATCH_SIZE):
        batch = X_latent_shuffled[i:i+BATCH_SIZE]
        som(batch, iter_count)
        iter_count += 1

# ==========================================
# 5. Visualization
# ==========================================
som.eval()

# Get BMU for every sample
with torch.no_grad():
    # Calculate distances
    dists = torch.cdist(X_latent, som.weights)
    _, bmu_indices = torch.min(dists, dim=1)
    bmu_indices = bmu_indices.cpu().numpy()

# Create a map for visualization
map_grid = np.zeros((SOM_WIDTH, SOM_HEIGHT))
# Dictionary to hold labels for each node
node_labels = {} 

for i, bmu_idx in enumerate(bmu_indices):
    x_coord = bmu_idx // SOM_HEIGHT
    y_coord = bmu_idx % SOM_HEIGHT
    
    label_name = le.inverse_transform([y_encoded[i]])[0]
    
    if (x_coord, y_coord) not in node_labels:
        node_labels[(x_coord, y_coord)] = []
    node_labels[(x_coord, y_coord)].append(label_name)

# Plotting
plt.figure(figsize=(8, 6))
ax = plt.gca()

# Draw som grid
for x in range(SOM_WIDTH + 1):
    ax.axvline(x, color='gray', linestyle='--', alpha=0.2)
for y in range(SOM_HEIGHT + 1):
    ax.axhline(y, color='gray', linestyle='--', alpha=0.2)

# Plot points with jitter to see overlapping samples
# Add random jitter
jitter_x = np.random.uniform(0.2, 0.8, size=len(bmu_indices))
jitter_y = np.random.uniform(0.2, 0.8, size=len(bmu_indices))

for i, bmu_idx in enumerate(bmu_indices):
    x = bmu_idx // SOM_HEIGHT
    y = bmu_idx % SOM_HEIGHT
    
    # Color based label
    color = plt.cm.tab10(y_encoded[i] / len(le.classes_))
    
    plt.plot(x + jitter_x[i], y + jitter_y[i], 
             marker='o', 
             color=color, 
             markersize=6, 
             alpha=0.7)

# Create custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.tab10(i/len(le.classes_)), label=l) 
           for i, l in enumerate(le.classes_)]
plt.legend(handles=handles, title="Disease Label", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title(f"Deep SOM Analysis on GSE33000\n(Autoencoder Latent Space -> SOM Grid)", pad=10)
plt.xlim(0, SOM_WIDTH)
plt.ylim(0, SOM_HEIGHT)
plt.gca().invert_yaxis() 
plt.tight_layout()
plt.show()

### Same code as above but ran again for generating a different grid

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Configuration 
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# SOM Hyperparameters
SOM_WIDTH = 10
SOM_HEIGHT = 10
LATENT_DIM = 64      # Compressed dimension (Autoencoder output)
AE_EPOCHS = 150       # Autoencoder pre-training epochs
SOM_EPOCHS = 100     # SOM training epochs
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

# ==========================================
# 2. Data  
# ==========================================
data_df = pd.read_csv('GSE33000_Top10000_Var.csv', index_col=0, engine='python')

# Separate features and labels
X_raw = data_df.iloc[:, :-1].values
y_raw = data_df.iloc[:, -1].values

y_raw = np.where((y_raw == 'AD') | (y_raw == 'HD'), 'A', y_raw)

# 1. Label Encoding 
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

# 2. Feature Scaling 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

dataset = TensorDataset(
    torch.FloatTensor(X_scaled).to(device), 
    torch.LongTensor(y_encoded).to(device)
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ==========================================
# 3. Deep Learning 
# ==========================================

class Autoencoder(nn.Module):
    """
    Compresses 10,000 features down to LATENT_DIM (e.g., 64).
    This extracts non-linear gene patterns before the SOM.
    """
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim) # Latent representation
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

class SOM(nn.Module):
    """
    PyTorch implementation of a Self-Organizing Map.
    Optimized for GPU batch processing.
    """
    def __init__(self, m, n, dim, n_iter, alpha=None, sigma=None):
        super(SOM, self).__init__()
        self.m = m
        self.n = n
        self.dim = dim
        self.n_iter = n_iter
        self.alpha = alpha if alpha else 0.3
        self.sigma = sigma if sigma else max(m, n) / 2.0
        
        # Initialize weights randomly
        self.weights = nn.Parameter(torch.randn(m * n, dim), requires_grad=False)
        
        # Pre-compute grid locations for neighborhood calculation
        self.locations = torch.LongTensor(np.array(list(self.neuron_locations()))).to(device)
        self.pdist = nn.PairwiseDistance(p=2)

    def neuron_locations(self):
        for i in range(self.m):
            for j in range(self.n):
                yield np.array([i, j])

    def forward(self, x, it):
        # x shape: (batch_size, dim)
        # weights shape: (num_neurons, dim)
        
        # 1. Calculate distances between inputs and weights
        dists = torch.cdist(x, self.weights) # Shape: (batch, num_neurons)
        
        # 2. Find Best Matching Units (BMU)
        _, bmu_indices = torch.min(dists, dim=1)
        
        # 3. Update Weights (if in training mode)
        if self.training:
            bmu_locs = self.locations[bmu_indices] # (batch, 2)
            
            # Decay parameters
            alpha_t = self.alpha * (1 - it / self.n_iter)
            sigma_t = self.sigma * (1 - it / self.n_iter)
            
            # Calculate neighborhood function for all neurons against all batch BMUs
            
            for i in range(x.size(0)): # Iterate batch
                bmu_loc = bmu_locs[i]
                input_vec = x[i]
                
                # Grid distances from BMU
                grid_dists = torch.sum((self.locations - bmu_loc).pow(2), dim=1).float()
                
                # Neighborhood function (Gaussian)
                neighborhood = torch.exp(-grid_dists / (2 * (sigma_t ** 2)))
                
                # Update rule: W = W + alpha * neighborhood * (X - W)
                # Influence shape: (num_neurons, 1)
                influence = (alpha_t * neighborhood).unsqueeze(1)
                self.weights.data += influence * (input_vec - self.weights.data)
                
        return bmu_indices

# ==========================================
# 4. Training 
# ==========================================

#  Phase 1: Train Autoencoder 
print(f"\nPhase 1: Training Autoencoder ({AE_EPOCHS} epochs)")
input_dim = X_scaled.shape[1]
ae_model = Autoencoder(input_dim, LATENT_DIM).to(device)
ae_optimizer = optim.Adam(ae_model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

for epoch in range(AE_EPOCHS):
    total_loss = 0
    for batch_x, _ in dataloader:
        ae_optimizer.zero_grad()
        _, decoded = ae_model(batch_x)
        loss = criterion(decoded, batch_x)
        loss.backward()
        ae_optimizer.step()
        total_loss += loss.item()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{AE_EPOCHS}, Loss: {total_loss / len(dataloader):.4f}")

#  Phase 2: Extract Latent Features 
ae_model.eval()
with torch.no_grad():
    X_tensor = torch.FloatTensor(X_scaled).to(device)
    X_latent, _ = ae_model(X_tensor)
    
#  Phase 3: Train SOM on Latent Features 
print(f"\nPhase 2: Training SOM ({SOM_EPOCHS} epochs)")
som = SOM(m=SOM_WIDTH, n=SOM_HEIGHT, dim=LATENT_DIM, n_iter=SOM_EPOCHS*len(dataloader)).to(device)
som.train()

iter_count = 0
for epoch in range(SOM_EPOCHS):
    # Shuffle latent data for SOM training
    indices = torch.randperm(X_latent.size(0))
    X_latent_shuffled = X_latent[indices]
    
    # Process in batches
    for i in range(0, X_latent.size(0), BATCH_SIZE):
        batch = X_latent_shuffled[i:i+BATCH_SIZE]
        som(batch, iter_count)
        iter_count += 1

# ==========================================
# 5. Visualization
# ==========================================
som.eval()

# Get BMU for every sample
with torch.no_grad():
    # Calculate distances
    dists = torch.cdist(X_latent, som.weights)
    _, bmu_indices = torch.min(dists, dim=1)
    bmu_indices = bmu_indices.cpu().numpy()

# Create a map for visualization
map_grid = np.zeros((SOM_WIDTH, SOM_HEIGHT))
# Dictionary to hold labels for each node
node_labels = {} 

for i, bmu_idx in enumerate(bmu_indices):
    x_coord = bmu_idx // SOM_HEIGHT
    y_coord = bmu_idx % SOM_HEIGHT
    
    label_name = le.inverse_transform([y_encoded[i]])[0]
    
    if (x_coord, y_coord) not in node_labels:
        node_labels[(x_coord, y_coord)] = []
    node_labels[(x_coord, y_coord)].append(label_name)

# Plotting
plt.figure(figsize=(8, 6))
ax = plt.gca()

# Draw som grid
for x in range(SOM_WIDTH + 1):
    ax.axvline(x, color='gray', linestyle='--', alpha=0.2)
for y in range(SOM_HEIGHT + 1):
    ax.axhline(y, color='gray', linestyle='--', alpha=0.2)

# Plot points with jitter to see overlapping samples
# Add random jitter
jitter_x = np.random.uniform(0.2, 0.8, size=len(bmu_indices))
jitter_y = np.random.uniform(0.2, 0.8, size=len(bmu_indices))

for i, bmu_idx in enumerate(bmu_indices):
    x = bmu_idx // SOM_HEIGHT
    y = bmu_idx % SOM_HEIGHT
    
    # Color based label
    color = plt.cm.tab10(y_encoded[i] / len(le.classes_))
    
    plt.plot(x + jitter_x[i], y + jitter_y[i], 
             marker='o', 
             color=color, 
             markersize=6, 
             alpha=0.7)

# Create custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.cm.tab10(i/len(le.classes_)), label=l) 
           for i, l in enumerate(le.classes_)]
plt.legend(handles=handles, title="Disease Label", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title(f"Deep SOM Analysis on GSE33000\n(Autoencoder Latent Space -> SOM Grid)", pad=10)
plt.xlim(0, SOM_WIDTH)
plt.ylim(0, SOM_HEIGHT)
plt.gca().invert_yaxis() 
plt.tight_layout()
plt.show()